In [1]:
# ========== 导入：多模型辩论聊天所需的库 ==========

# os：读环境变量（Environment Variables），例如各家 API Key
import os
# requests：用 HTTP 探测本地 Ollama 是否在跑
import requests
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# OpenAI 客户端：也用于 OpenRouter / Ollama / Gemini 的 OpenAI 兼容端点
from openai import OpenAI
# Markdown + display：在 Jupyter 里渲染对话记录
from IPython.display import Markdown, display
# gradio：快速搭 Web 辩论界面
import gradio as gr


In [ ]:
# ========== 环境：加载 .env 并检查三家密钥是否存在 ==========

# override=True：以 .env 文件覆盖进程里已有同名变量
load_dotenv(override=True)
# 从环境读取 OpenAI 云端密钥
openai_api_key = os.getenv('OPENAI_API_KEY')
# Google / Gemini 密钥（本笔记本里可选）
google_api_key = os.getenv('GOOGLE_API_KEY')
# OpenRouter 聚合网关密钥（Blake 用 openrouter）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


# 有密钥则打印前缀做快速确认（勿打印完整密钥）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# ========== 客户端：同一套 OpenAI SDK，换 base_url 指向不同后端 ==========

# 默认客户端（读环境里的 OPENAI_API_KEY）；后面实际多用显式 openai_client
openai = OpenAI()

# 理念：Gemini / OpenRouter / Ollama 都提供 OpenAI 兼容 HTTP 端点
# 所以可以复用 openai.OpenAI，只改 api_key 与 base_url


# OpenAI 官方云端
openai_client = OpenAI(api_key=openai_api_key)
# OpenRouter：统一入口调多家模型
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
# 本地 Ollama：api_key 多为占位字符串 "ollama"
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
# Gemini 的 OpenAI 兼容端点（需 GOOGLE_API_KEY）
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")


In [ ]:
# ========== 探测：确认本机 Ollama HTTP 服务是否可达 ==========

# 访问 Ollama 根路径；有响应说明服务在跑（内容本身不重要）
requests.get("http://localhost:11434/").content


In [ ]:
# ========== 角色配置：三个 Agent 的人设 / 供应商 / 模型 ==========

# AGENTS：每人一个 dict——name、provider、model、system（发给模型的人设，保留英文）
AGENTS = [
    {
        "name": "Alex",
        "provider": "openai",
        "model": "gpt-4.1-mini",
        "system": "You are Alex. You are argumentative, skeptical, and slightly snarky. You challenge claims directly."
    },
    {
        "name": "Blake",
        "provider": "openrouter",
        "model": "openrouter/aurora-alpha",
        "system": "You are Blake. You are polite and cooperative. You look for common ground and de-escalate tension."
    },
    {
        "name": "Charles",
        "provider": "ollama",
        "model": "llama3.2",
        "system": "You are Charles. You are naive, literal, and often miss the point. You drift into misunderstandings."
    }
]

# provider 字符串 → 上面创建好的客户端实例
CLIENTS = {
    "openai": openai_client,
    "openrouter": openrouter_client,
    "ollama": ollama_client,
}

def validate_config(agents):
    # 根据实际用到的 provider，提前检查对应密钥是否存在
    providers = {a["provider"] for a in agents}
    if "openai" in providers and not openai_api_key:
        raise ValueError("OPENAI_API_KEY is required for provider 'openai'")
    if "openrouter" in providers and not openrouter_api_key:
        raise ValueError("OPENROUTER_API_KEY is required for provider 'openrouter'")

# 启动前校验，缺密钥就立刻失败，避免辩论中途报错
validate_config(AGENTS)


In [ ]:
# ========== 核心逻辑：拼 messages、生成回复、多轮辩论（含流式） ==========

def build_messages(current_agent, transcript):
    # 当前发言者的 system 人设放第一条
    messages = [{"role": "system", "content": current_agent["system"]}]

    # 把 transcript 转成「对当前 agent 视角」的 chat history
    for turn in transcript:
        speaker = turn["speaker"]
        text = turn["text"]

        # 自己说过的话 → assistant；别人说的 → user（带说话人名字）
        if speaker == current_agent["name"]:
            messages.append({"role": "assistant", "content": text})
        else:
            messages.append({"role": "user", "content": f"{speaker}: {text}"})

    return messages

def generate_reply(agent, transcript, temperature=0.7):
    # 按 provider 选客户端
    client = CLIENTS[agent["provider"]]
    # 拼完整 messages
    messages = build_messages(agent, transcript)

    # 非流式一次拿完整回复
    response = client.chat.completions.create(
        model=agent["model"],
        messages=messages,
        temperature=temperature
    )

    # content 可能为 None，统一 strip 成字符串
    return (response.choices[0].message.content or "").strip()

def generate_reply_stream(agent, transcript, temperature=0.7):
    # 流式版：边生成边 yield 累计全文，方便 Gradio 打字机效果
    client = CLIENTS[agent["provider"]]
    messages = build_messages(agent, transcript)

    stream = client.chat.completions.create(
        model=agent["model"],
        messages=messages,
        temperature=temperature,
        stream=True
    )

    full_text = ""
    for chunk in stream:
        # 有些 chunk 没有 choices，跳过
        if not getattr(chunk, "choices", None):
            continue
        delta = chunk.choices[0].delta
        if delta is None:
            continue
        piece = delta.content or ""
        if piece:
            full_text += piece
            # 每次 yield 到目前为止的完整文本
            yield full_text

def run_conversation(seed_prompt, rounds=3, verbose=True):
    # 非流式辩论：先写入用户种子话题
    transcript = [{"speaker": "User", "text": seed_prompt}]

    # rounds 轮 × 每个 AGENTS 依次发言
    for _ in range(rounds):
        for agent in AGENTS:
            reply = generate_reply(agent, transcript)
            transcript.append({"speaker": agent["name"], "text": reply})

            if verbose:
                print(f"{agent['name']}: {reply}\n")

    return transcript

def run_conversation_stream(seed_prompt, rounds=3, verbose=False):
    # 流式辩论：每有部分回复就 yield 整份 transcript（给 Gradio 刷新）
    transcript = [{"speaker": "User", "text": seed_prompt}]
    yield transcript

    for _ in range(rounds):
        for agent in AGENTS:
            # 先占位一条空回复，流式更新其 text
            transcript.append({"speaker": agent["name"], "text": ""})

            # transcript[:-1]：生成时不把自己这条空记录喂回去
            for partial_reply in generate_reply_stream(agent, transcript[:-1]):
                transcript[-1]["text"] = partial_reply
                yield transcript

            # 模型没返回任何字时的占位文案（保留英文）
            if not transcript[-1]["text"]:
                transcript[-1]["text"] = "[No response returned]"
                yield transcript

            if verbose:
                # 原文此处是字面 \n（两个字符），不是换行转义；保持不改
                print(f"{agent['name']}: {transcript[-1]['text']}\\n")

def render_markdown_transcript(transcript):
    # Jupyter 里把对话转成 Markdown 并 display
    parts = ["## Conversation Transcript"]
    for turn in transcript:
        parts.append(f"**{turn['speaker']}**: {turn['text']}")
    display(Markdown("\n\n".join(parts)))


In [ ]:
# ========== 可选：在笔记本里直接跑非流式辩论（默认整格注释掉） ==========

# 种子话题示例（英文可运行字符串；下面调用保持注释，避免一运行就打 API）
# seed = "Debate whether AI will improve education in the next 5 years."
# transcript = run_conversation(seed_prompt=seed, rounds=2, verbose=True)
# render_markdown_transcript(transcript)


In [ ]:
# ========== Gradio UI：流式展示三方辩论 ==========

def transcript_to_markdown(transcript):
    # 把 list[dict] 转成 Markdown 字符串（给 Interface 的 outputs）
    parts = ["## Conversation Transcript"]
    for turn in transcript:
        parts.append(f"**{turn['speaker']}**: {turn['text']}")
    return "\n\n".join(parts)

def debate_ui_stream(seed_prompt):
    # Gradio 生成器：每 yield 一次就刷新 Markdown
    try:
        for transcript in run_conversation_stream(seed_prompt=seed_prompt, rounds=2, verbose=False):
            yield transcript_to_markdown(transcript)
    except Exception as e:
        # 出错时把异常类型与信息显示在界面上（文案结构保留）
        yield f"## Error\n\n{type(e).__name__}: {e}"

# Interface：输入话题 → 流式输出 Markdown 对话
view = gr.Interface(
    fn=debate_ui_stream,
    title="Model Debates",
    inputs=gr.Textbox(label="Proposed Topic", info="Enter a debate topic", lines=4),
    outputs=gr.Markdown(label="Response"),
    examples=[["Debate whether AI will improve education in the next 5 years."]],
    flagging_mode="never"
)

# queue()：排队支持并发流式；share=True 会尝试生成公网链接
view.queue().launch(share=True)
